# Full pipeline — one-shot Colab run (full dataset)

Runs everything downstream of data prep in order: **02 baselines → 03 MF-BPR →
04 two-tower → 05 LightGBM ranking**, then exports webapp `artifacts/` and syncs
results back to Drive.

## Before you run
1. **Runtime → Change runtime type → T4 GPU** (used by the two-tower in section 04;
   sections 02/03 are CPU-bound by design).
2. **Upload the FULL prepared dataset to Drive** at
   `MyDrive/amazon-recsys-data/data/` — that's the local `data/` folder containing
   `train/val/test/items.parquet` **with the encoded `u`/`i` columns** and the
   matching `id_encoders.json` (551,911 users / 148,177 items).
   ⚠️ The repo's committed `data/` files are the old ~131K-interaction sample —
   the sanity-check cell below will stop you if the full data isn't found.
3. Run all (Runtime → Run all). Rough timings on Colab: 02 ≈ 30–60 min,
   03 ≈ 1–1.5 h (CPU by design), 04 ≈ 10–20 min on T4, 05 ≈ 30–60 min.

Data prep (notebook 01) is intentionally **not** re-run here: the full dataset is
already built, and re-streaming the raw 22.6 GB file produces a slightly different
dataset (breaking comparability with the frozen split). Set `RUN_DATA_PREP = True`
in the optional cell below only if you really need to rebuild from raw.

In [ ]:
import os, subprocess
from pathlib import Path

def _find_repo():
    """Repo root = a dir containing src/ + requirements.txt. Checks the cwd,
    an existing clone next to it, and the parent (launched from notebooks/)."""
    for p in [Path.cwd(), Path.cwd() / "amazon-recsys", Path("/content/amazon-recsys"), Path.cwd().parent]:
        if (p / "src").exists() and (p / "requirements.txt").exists():
            return p
    return None

repo = _find_repo()
if repo is None:
    target = Path("amazon-recsys")
    if target.exists():                     # leftover non-repo dir: clone beside it, don't touch it
        target = Path("amazon-recsys-repo")
    r = subprocess.run(["git", "clone", "https://github.com/21zaimotman-tech/amazon-recsys.git", str(target)],
                       capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f"git clone failed:\n{r.stderr}")
    repo = target
os.chdir(repo)
print("cwd:", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt

## Pin the fixed retrieval-scoring code

The two cells below overwrite `src/models/mf_bpr.py` and `src/eval/harness.py`
with the versions carrying the **item-bias folding fix** (FAISS retrieval must
reproduce the exact training-time score `dot(user,item) + item_bias`; the version
on GitHub drops the bias and L2-normalizes). Harmless if the repo is already
up to date — the files are simply rewritten with identical content.

In [ ]:
%%writefile src/models/mf_bpr.py
"""Matrix Factorization with BPR pairwise loss (PyTorch), trained with SGD.

================================ READ THIS ================================
The brief requires you to implement this FROM SCRATCH and EXPLAIN EVERY LINE
at the defense. Use this as a reference for the structure, then make sure each
of you can derive the BPR loss and explain the sampling. Do not submit code you
cannot defend — the examiners will ask.
==========================================================================

BPR idea: for a user u, an observed (positive) item i should score higher than
an unobserved (negative) item j. We optimise the pairwise ranking:

    L = - sum over (u,i,j) of  log sigmoid( s_ui - s_uj )  +  reg * ||params||^2

where s_ui = <p_u, q_i> + b_i  is the predicted score (dot product of user and
item embeddings plus an item bias).
"""
from __future__ import annotations
import numpy as np
import torch
import torch.nn as nn


class MFBPR(nn.Module):
    def __init__(self, n_users, n_items, dim=64, reg=1e-5):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, dim)
        self.item_emb = nn.Embedding(n_items, dim)
        self.item_bias = nn.Embedding(n_items, 1)
        nn.init.normal_(self.user_emb.weight, std=0.01)
        nn.init.normal_(self.item_emb.weight, std=0.01)
        nn.init.zeros_(self.item_bias.weight)
        self.reg = reg

    def score(self, u, i):
        return (self.user_emb(u) * self.item_emb(i)).sum(-1) + self.item_bias(i).squeeze(-1)

    def bpr_loss(self, u, i, j):
        """Pairwise BPR loss for a batch of (user, pos_item, neg_item) triplets.

        ----- THE CORE LINES YOU MUST BE ABLE TO EXPLAIN -----
        s_ui = self.score(u, i)
        s_uj = self.score(u, j)
        # maximise margin (s_ui - s_uj) => minimise -log(sigmoid(margin))
        loss = -torch.log(torch.sigmoid(s_ui - s_uj) + 1e-9).mean()
        # L2 regularisation on the embeddings used in this batch
        reg = self.reg * (self.user_emb(u).pow(2).sum()
                          + self.item_emb(i).pow(2).sum()
                          + self.item_emb(j).pow(2).sum())
        return loss + reg / len(u)
        ------------------------------------------------------
        """
        s_ui = self.score(u, i)
        s_uj = self.score(u, j)
        loss = -torch.log(torch.sigmoid(s_ui - s_uj) + 1e-9).mean()
        reg = self.reg * (self.user_emb(u).pow(2).sum()
                          + self.item_emb(i).pow(2).sum()
                          + self.item_emb(j).pow(2).sum())
        return loss + reg / len(u)


def sample_triplets(train_df, n_items, rng, batch_size, pos=None, user_items=None):
    """Uniformly sample (u, pos_i, neg_j). neg_j is a random item the user has
    not interacted with. BONUS: try popularity-based negative sampling and
    compare (see the bonus list).

    `pos`/`user_items` can be precomputed once by the caller and passed in —
    rebuilding `user_items` (a groupby + per-user set) from scratch on every
    call is fine for a handful of calls, but train_mfbpr calls this once per
    *step* (thousands of times per epoch), where re-deriving it every time
    dominates the runtime and starves the GPU of work. Passing it in once
    keeps the actual sampling logic below identical."""
    if pos is None:
        pos = train_df[["u", "i"]].values
    if user_items is None:
        user_items = train_df.groupby("u")["i"].agg(set).to_dict()
    idx = rng.integers(0, len(pos), size=batch_size)
    u = pos[idx, 0]
    i = pos[idx, 1]
    j = rng.integers(0, n_items, size=batch_size)
    for k in range(batch_size):           # resample collisions with positives
        while j[k] in user_items[u[k]]:
            j[k] = rng.integers(0, n_items)
    return u, i, j


def train_mfbpr(train_df, n_users, n_items, dim=64, lr=0.05, reg=1e-5,
                epochs=20, batch_size=4096, steps_per_epoch=2000, device="cpu"):
    rng = np.random.default_rng(0)
    model = MFBPR(n_users, n_items, dim, reg).to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    pos = train_df[["u", "i"]].values                             # built once, not per step
    user_items = train_df.groupby("u")["i"].agg(set).to_dict()    # built once, not per step
    for ep in range(epochs):
        model.train(); running = 0.0
        for _ in range(steps_per_epoch):
            u, i, j = sample_triplets(train_df, n_items, rng, batch_size, pos, user_items)
            u = torch.as_tensor(u, device=device); i = torch.as_tensor(i, device=device)
            j = torch.as_tensor(j, device=device)
            opt.zero_grad()
            loss = model.bpr_loss(u, i, j)
            loss.backward(); opt.step()
            running += loss.item()
        print(f"epoch {ep+1}/{epochs}  loss={running/steps_per_epoch:.4f}")
    return model


def export_item_embeddings(model):
    """Item matrix for FAISS = item_emb, with item_bias folded in as an extra
    dimension (so raw dot product with an augmented user vector reproduces the
    exact training-time score: dot(user,item) + item_bias)."""
    item_vecs = model.item_emb.weight.detach().cpu().numpy()
    item_bias = model.item_bias.weight.detach().cpu().numpy()  # (n_items, 1)
    return np.concatenate([item_vecs, item_bias], axis=1)


In [ ]:
%%writefile src/eval/harness.py
"""Batch evaluation harness — the glue between a trained model and metrics.py.

A retriever gives you item embeddings + a way to produce a user vector. This
module turns that into a {user -> ranked items} dict (excluding items the user
already saw in training) and scores it. Used by notebooks 02-05.
"""
from __future__ import annotations
import numpy as np
from .metrics import evaluate


def seen_items(train_df) -> dict:
    """{user_id -> set(item_id)} seen during training. Excluded from recs so we
    don't get credit for re-recommending known items (and metrics stay honest)."""
    return train_df.groupby("user_id")["item_id"].agg(set).to_dict()


def recommend_from_index(index, user_vecs, user_ids, seen=None, n=50, retrieve=200):
    """index: EmbeddingIndex. user_vecs: (U, dim) aligned with user_ids.
    Retrieve `retrieve` candidates, drop training-seen items, keep top-n."""
    seen = seen or {}
    raw = index.search(np.asarray(user_vecs, dtype="float32"), retrieve)
    recs = {}
    for uid, items in zip(user_ids, raw):
        s = seen.get(uid, ())
        recs[uid] = [it for it in items if it not in s][:n]
    return recs


def evaluate_index(index, user_vecs, user_ids, test_gt, train_df, catalog_size,
                   k_values=(10, 20, 50), retrieve=200):
    """One call: recommend for all test users, then score. Returns (recs, metrics)."""
    recs = recommend_from_index(index, user_vecs, user_ids, seen_items(train_df),
                                n=max(k_values), retrieve=retrieve)
    return recs, evaluate(recs, test_gt, catalog_size, k_values)


# ---- user-vector builders (call from the notebook) ----
def mfbpr_user_vecs(model, user_idx):
    """MF-BPR user vectors = rows of the user embedding table, with a constant
    1 appended so the dot product with export_item_embeddings' augmented item
    vectors reproduces dot(user,item) + item_bias exactly."""
    vecs = model.user_emb.weight.detach().cpu().numpy()[user_idx]
    ones = np.ones((vecs.shape[0], 1), dtype=vecs.dtype)
    return np.concatenate([vecs, ones], axis=1)


def two_tower_user_vecs(model, histories, max_hist=20, device="cpu"):
    """histories: list[list[int encoded item ids]] (each user's TRAIN history,
    or TEST-period history when simulating serving). Returns (U, dim) normalised."""
    import torch
    import torch.nn.functional as F
    vecs = []
    model.eval()
    with torch.no_grad():
        for h in histories:
            h = h[-max_hist:]
            if not h:
                vecs.append(np.zeros(model.item_tower.emb.embedding_dim, dtype="float32"))
                continue
            t = torch.tensor([h], device=device)
            m = torch.ones_like(t, dtype=torch.float32)
            v = F.normalize(model.user_tower(t, m), dim=-1)[0].cpu().numpy()
            vecs.append(v)
    return np.vstack(vecs)


## Restore data from Drive + sanity-check it is the FULL encoded dataset

In [ ]:
from pathlib import Path
DATA = Path("data"); DATA.mkdir(exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DATA = Path('/content/drive/MyDrive/amazon-recsys-data/data')
    if (DRIVE_DATA / "train.parquet").exists():
        import shutil
        shutil.copytree(DRIVE_DATA, DATA, dirs_exist_ok=True)
        print(f"restored data/ from {DRIVE_DATA}")
    else:
        print(f"nothing at {DRIVE_DATA} — relying on whatever is already in data/")
except ImportError:
    pass   # not running in Colab

In [ ]:
# HARD STOP if data/ is not the full, encoded dataset. A fresh git clone brings
# the old ~131K-row SAMPLE files committed to the repo; training on those (or on
# parquets whose u/i encoding doesn't match id_encoders.json) silently produces
# garbage. Fix: upload the local full data/ folder to MyDrive/amazon-recsys-data/data.
import json
import pandas as pd

_train = pd.read_parquet(DATA / "train.parquet")
_ids = json.load(open(DATA / "id_encoders.json"))
assert "u" in _train.columns and "i" in _train.columns, (
    "train.parquet has no u/i columns — this is the raw build_dataset.py output, "
    "not the encoded dataset. Upload the encoded data/ folder to Drive.")
assert len(_train) > 3_000_000, (
    f"train.parquet has only {len(_train):,} rows — this is the SAMPLE dataset. "
    "Upload the FULL data/ folder to MyDrive/amazon-recsys-data/data and re-run.")
assert _train["u"].max() < len(_ids["user_ids"]) and _train["i"].max() < len(_ids["item_ids"]), (
    "id_encoders.json does not match train.parquet's u/i encoding — they must come "
    "from the same run. Upload the matching pair from the local data/ folder.")
print(f"OK: full dataset — train={len(_train):,} rows, "
      f"n_users={len(_ids['user_ids']):,}, n_items={len(_ids['item_ids']):,}")
del _train

## Optional: rebuild the dataset from raw (default OFF)

Leave `RUN_DATA_PREP = False` unless you deliberately want to re-stream the raw
~22.6 GB reviews file and rebuild everything (≈30–45 min, and the result will
differ slightly from the frozen split every run).

In [ ]:
RUN_DATA_PREP = False

if RUN_DATA_PREP:
    import json, time
    from scripts.build_dataset import stream_reviews, stream_meta
    from src.data.load import subsample, encode_ids
    from src.data.split import time_split, warn_leakage

    t0 = time.time()
    df = stream_reviews(limit=None, local_file=None)     # full stream from HuggingFace
    print(f"raw kept: {len(df):,} in {time.time()-t0:.0f}s")
    df = subsample(df)
    train_df, val_df, test_df = time_split(df)
    warn_leakage(train_df, test_df)
    meta = stream_meta(df.item_id.unique(), None, None)
    print(f"items with metadata: {len(meta):,} / {df.item_id.nunique():,}")
    train_df, val_df, test_df, user_ids, item_ids = encode_ids(train_df, val_df, test_df)
    train_df.to_parquet(DATA / "train.parquet")
    val_df.to_parquet(DATA / "val.parquet")
    test_df.to_parquet(DATA / "test.parquet")
    meta.to_parquet(DATA / "items.parquet")
    json.dump({"user_ids": user_ids, "item_ids": item_ids}, open(DATA / "id_encoders.json", "w"))
    print("saved -> data/{train,val,test,items,id_encoders.json}")
else:
    print("RUN_DATA_PREP=False — using the dataset restored above")

---
# Section 2 — Baselines (Random, Popularity)
*(cells from `notebooks/02_baselines.ipynb`)*

# 02 — Baselines

**Owner:** Person A · **Course session:** S1 (Foundations)

Loads the frozen split from Notebook 01 and scores two baselines through the shared
`src.eval.metrics.evaluate` — the same function every later notebook uses, so every table in
this project is comparable.

- **Random** — the floor. If a real model can't beat this, it isn't learning anything.
- **Popularity** — the real baseline to beat. On a head-concentrated catalog like Electronics
  this is a *strong* baseline on Recall/NDCG, and that is the interesting part.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, ".." if Path("../src").exists() else ".")

import pandas as pd

from src import config as C
from src.baselines.popularity import PopularityRecommender, RandomRecommender
from src.data.split import ground_truth_from
from src.eval.metrics import evaluate

DATA = Path("../data") if Path("../data").exists() else Path("data")
DATA.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DATA = Path('/content/drive/MyDrive/amazon-recsys-data/data')
    if (DRIVE_DATA / "train.parquet").exists():
        import shutil
        shutil.copytree(DRIVE_DATA, DATA, dirs_exist_ok=True)
        print(f"restored data/ from Google Drive ({DRIVE_DATA}) -- this takes priority over "
              f"whatever a fresh git clone brought in, since Drive holds the last real run's "
              f"actual output, not just whatever was last committed to GitHub")
except ImportError:
    pass   # not running in Colab (e.g. local smoke-test)

train = pd.read_parquet(DATA / "train.parquet")
test = pd.read_parquet(DATA / "test.parquet")
print(f"train={len(train):,}  test={len(test):,}")

## Eval setup

- **Catalog** = distinct items seen in train (the recommendable universe).
- **Ground truth** = test-period positives (`rating >= 4`), one set per user.
- **Exclusion** = each user's train-seen items are removed from their candidate list before
  scoring, so a model can't get credit for re-recommending something the user already has.

In [ ]:
catalog = train.item_id.nunique()
test_gt = ground_truth_from(test, positive_only=True)
users = list(test_gt.keys())
seen = train.groupby("user_id")["item_id"].agg(set).to_dict()
N = max(C.K_VALUES)

print(f"catalog={catalog:,}  test users scored={len(users):,}  k_values={C.K_VALUES}")

In [ ]:
rows = []
for name, Model in [("Random", RandomRecommender), ("Popularity", PopularityRecommender)]:
    model = Model().fit(train)
    recs = model.recommend(users, n=N, exclude=seen)
    m = evaluate(recs, test_gt, catalog_size=catalog, k_values=C.K_VALUES)
    m["method"] = name
    rows.append(m)

baseline_results = pd.DataFrame(rows).set_index("method")
report_cols = [c for c in baseline_results.columns if c != "n_users_scored"]
baseline_results[report_cols].round(4)

## Interpretation

**Popularity's Recall/NDCG is far above Random's, but its Coverage is close to zero** (it
recommends the same handful of head items to every user, so across all users it only ever
touches a tiny slice of the catalog) **while Random's Coverage is ~1.0** (given enough users,
a uniform-random pick eventually samples almost the whole catalog, it just never lands on the
*right* item for a given user). That's the required contrast for `ANALYSIS.md` §1: Popularity
wins on accuracy exactly because Electronics reviews concentrate on a small set of items, and
loses completely on diversity for the same reason. Any retriever we build next (MF-BPR,
two-tower) has to beat Popularity's Recall/NDCG *and* do meaningfully better than its Coverage
to be worth serving — beating Popularity on Recall alone by memorizing the head is not a win.

These two rows are the first two entries in the final comparison table (`ANALYSIS.md` §1);
Notebook 03 appends MF-BPR to it.

In [ ]:
baseline_results[report_cols].round(4).to_csv(DATA / "baseline_results.csv")
print("saved -> data/baseline_results.csv")

In [ ]:
try:
    from google.colab import drive
    import shutil
    DRIVE_DATA = Path('/content/drive/MyDrive/amazon-recsys-data/data')
    DRIVE_DATA.mkdir(parents=True, exist_ok=True)
    shutil.copy(DATA / "baseline_results.csv", DRIVE_DATA / "baseline_results.csv")
    print(f"synced baseline_results.csv -> Google Drive ({DRIVE_DATA})")
except ImportError:
    pass

---
# Section 3 — Retrieval: MF-BPR
*(cells from `notebooks/03_retrieval_mfbpr.ipynb`)*

# 03 — Retrieval: MF-BPR

**Owner:** Person B · **Course session:** S2 (Matrix Factorization) · **25% of grade with Notebook 04**

Trains Matrix Factorization with a pairwise BPR loss from scratch (`src/models/mf_bpr.py`),
tunes on **val only**, builds a FAISS index over the learned item embeddings, and evaluates
retrieval quality against the Notebook 02 baselines.

> **Colab setup:** Runtime → Change runtime type → GPU (T4) is not required for MF-BPR (it's
> cheap — CPU is fine and saves GPU quota for Notebook 04's two-tower), but harmless either way.
> Mount Drive and point `DATA` below at your copy of Notebook 01's `data/` output if you saved
> it there instead of running Notebook 01 fresh in this session.

**Before the defense, every team member must be able to derive `bpr_loss` line-by-line and
explain why `sample_triplets` resamples a negative that collides with a user's positives** —
these are called out explicitly in `src/models/mf_bpr.py`'s docstring as defense questions.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS: torch+faiss both bundle OpenMP and abort on double-init otherwise
import sys, time, json
from pathlib import Path
sys.path.insert(0, ".." if Path("../src").exists() else ".")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config as C
from src.models.mf_bpr import MFBPR, sample_triplets, export_item_embeddings
from src.retrieval.faiss_index import EmbeddingIndex
from src.eval.harness import evaluate_index, mfbpr_user_vecs
from src.data.split import ground_truth_from

plt.rcParams["figure.dpi"] = 110
# MF-BPR is tiny (dim=64, a single dot-product score) and trains in thousands of small,
# sequential steps -- GPU kernel-launch overhead (plus the .item() sync every step) makes
# a T4 slower than CPU here, not faster. Force CPU regardless of Runtime type selected;
# save the GPU for Notebook 04's two-tower, which actually benefits from it.
DEVICE = "cpu"
print("device:", DEVICE, "(forced -- MF-BPR does not benefit from GPU, see comment above)")

DATA = Path("../data") if Path("../data").exists() else Path("data")
DATA.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DATA = Path('/content/drive/MyDrive/amazon-recsys-data/data')
    if (DRIVE_DATA / "train.parquet").exists():
        import shutil
        shutil.copytree(DRIVE_DATA, DATA, dirs_exist_ok=True)
        print(f"restored data/ from Google Drive ({DRIVE_DATA}) -- this takes priority over "
              f"whatever a fresh git clone brought in, since Drive holds the last real run's "
              f"actual output, not just whatever was last committed to GitHub")
except ImportError:
    pass   # not running in Colab (e.g. local smoke-test)

train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
test = pd.read_parquet(DATA / "test.parquet")
ids = json.load(open(DATA / "id_encoders.json"))
user_ids, item_ids = ids["user_ids"], ids["item_ids"]
uid2u = {u: i for i, u in enumerate(user_ids)}       # raw user_id -> encoded u (train vocab only)
n_users, n_items = len(user_ids), len(item_ids)
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}  n_users={n_users:,}  n_items={n_items:,}")

baseline_results = pd.read_csv(DATA / "baseline_results.csv", index_col=0)
baseline_results

## Id spaces, and why they matter here

Two different id spaces are in play and mixing them up silently breaks the eval:

- **Encoded ints (`u`/`i` columns)** — only used to index into the model's embedding tables
  (`model.user_emb`, `model.item_emb`). `encode_ids` (Notebook 01) reserves a trailing
  `<UNK>` row (index `n_users`/`n_items`) for a val/test id never seen in train, so embedding
  tables are built with `n_users + 1` / `n_items + 1` rows.
- **Raw string ids (`user_id`/`item_id`)** — everything from `src.eval` (`seen_items`,
  `ground_truth_from`, `evaluate`) and `EmbeddingIndex.search` operates in this space, exactly
  like the Notebook 02 baselines. `FAISS` is built with `item_ids` (the index → string list) as
  labels, so `index.search(...)` already returns raw string item ids.

So: encode only to fetch a user's embedding row, then immediately hand raw string ids to
everything eval-related below — never compare an encoded int against a raw id.

## Training loop (with loss history, for the required loss-curve plot)

Identical to `src.models.mf_bpr.train_mfbpr` — same triplet sampling, same `bpr_loss` call —
just additionally records the per-epoch mean loss for plotting. This *is* the training
function used everywhere below; there is no separate/duplicate training run.

In [ ]:
def train_mfbpr_with_history(train_df, n_users, n_items, dim, lr, reg, epochs,
                             batch_size, steps_per_epoch, device):
    """n_users/n_items here already include the +1 UNK row. `sample_triplets` still
    samples negatives only from the n_items-1 real (non-UNK) items.

    `pos`/`user_items` are built ONCE here, not inside the step loop: sample_triplets
    is called steps_per_epoch * epochs times (thousands), and rebuilding a
    groupby(\'u\')[\'i\'].agg(set).to_dict() over the full train set on every single
    call (rather than once per training run) is the actual bottleneck on a GPU
    runtime -- the GPU sits idle while the CPU redoes that pandas groupby between
    steps. Precomputing it once cuts training time by roughly steps_per_epoch x."""
    rng = np.random.default_rng(0)
    m = MFBPR(n_users, n_items, dim, reg).to(device)
    opt = torch.optim.SGD(m.parameters(), lr=lr)
    pos = train_df[["u", "i"]].values
    user_items = train_df.groupby("u")["i"].agg(set).to_dict()
    history = []
    for ep in range(epochs):
        m.train(); running = 0.0
        for _ in range(steps_per_epoch):
            u, i, j = sample_triplets(train_df, n_items - 1, rng, batch_size, pos, user_items)
            u = torch.as_tensor(u, device=device); i = torch.as_tensor(i, device=device)
            j = torch.as_tensor(j, device=device)
            opt.zero_grad(); loss = m.bpr_loss(u, i, j); loss.backward(); opt.step()
            running += loss.item()
        history.append(running / steps_per_epoch)
        print(f"epoch {ep+1}/{epochs}  loss={history[-1]:.4f}")
    return m, history


def retrieval_metrics(model, eval_df, train_df, catalog_size, k_values):
    """Encode eval_df's users just long enough to fetch embedding rows, then evaluate
    entirely in raw-string-id space (see markdown above) via the shared eval harness."""
    gt = ground_truth_from(eval_df, positive_only=True)          # raw uid -> set(raw item_id)
    users = [uid for uid in gt if uid in uid2u]                  # drop cold (train-unseen) users
    idx_rows = [uid2u[uid] for uid in users]
    uvecs = mfbpr_user_vecs(model, idx_rows)
    index = EmbeddingIndex(export_item_embeddings(model)[:n_items], item_ids, normalize=False)  # UNK row excluded; raw dot product to match training score (bias folded in)
    recs, metrics = evaluate_index(index, uvecs, users, gt, train_df, catalog_size, k_values)
    return recs, metrics

## Hyperparameter search (val only)

Small grid over `dim` / `lr` / `reg`, each run short (300 steps — just enough to rank
configs, not to converge). **Never touches `test` here.**

In [ ]:
grid = [
    dict(dim=32, lr=0.05, reg=1e-5),
    dict(dim=64, lr=0.05, reg=1e-5),
    dict(dim=64, lr=0.02, reg=1e-4),
]
search_rows = []
for cfg in grid:
    m, _ = train_mfbpr_with_history(train, n_users + 1, n_items + 1, cfg["dim"], cfg["lr"],
                                    cfg["reg"], epochs=1, batch_size=2048,
                                    steps_per_epoch=300, device=DEVICE)
    _, m20 = retrieval_metrics(m, val, train, catalog_size=n_items, k_values=(20,))
    search_rows.append({**cfg, "val_Recall@20": m20["Recall@20"]})

search_df = pd.DataFrame(search_rows).sort_values("val_Recall@20", ascending=False)
search_df

**Pick the best row above** and train the final, longer model with it (more
epochs/steps than the search needed — the search only had to *rank* configs).

In [ ]:
BEST = search_df.iloc[0][["dim", "lr", "reg"]].to_dict()
BEST["dim"] = int(BEST["dim"])
EPOCHS = 20
print("training with:", BEST)

t0 = time.time()
model, loss_history = train_mfbpr_with_history(train, n_users + 1, n_items + 1, BEST["dim"],
                                                BEST["lr"], BEST["reg"], EPOCHS,
                                                batch_size=4096, steps_per_epoch=2000,
                                                device=DEVICE)
print(f"trained in {time.time()-t0:.0f}s")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(range(1, len(loss_history) + 1), loss_history, color="#3b6fa0", marker="o")
ax.set_xlabel("epoch"); ax.set_ylabel("mean BPR loss"); ax.set_title("MF-BPR training loss")
plt.tight_layout(); plt.show()

## Retrieval evaluation (test)

Same `retrieval_metrics` helper, now on the held-out test split.

In [ ]:
recs, mfbpr_metrics = retrieval_metrics(model, test, train, catalog_size=n_items, k_values=C.K_VALUES)
mfbpr_metrics

In [ ]:
# Load comparison_results.csv if it already exists (Notebook 04/05 may have already
# added their rows) rather than baseline_results.csv fresh -- otherwise re-running this
# notebook after 04/05 silently wipes their rows back out.
results_path = DATA / "comparison_results.csv"
results = pd.read_csv(results_path, index_col=0) if results_path.exists() else baseline_results.copy()
results.loc["MF-BPR"] = {k: v for k, v in mfbpr_metrics.items() if k != "n_users_scored"}
results.round(4)

## Interpretation

Compare MF-BPR's row against Popularity's: MF-BPR personalizes — each user gets a *different*
top-N based on their own history — which should buy back Coverage even where raw Recall@k
is close to or below Popularity's on a very sparse, head-heavy catalog like this one (Popularity's
Recall/NDCG is hard to beat purely because so much test-period engagement concentrates on the
same handful of items). Where MF-BPR should visibly win is **Coverage**: it recommends
different items to different users, so across all users it touches far more of the catalog
than Popularity's one fixed list. This is an accuracy/diversity trade-off worth stating
explicitly in `ANALYSIS.md`, not a simple "MF-BPR wins."

**Defense checklist:** derive `bpr_loss` (`-log sigmoid(s_ui - s_uj) + reg`) line-by-line, and
explain why `sample_triplets`'s collision-resample loop matters — a negative that's secretly a
positive would train the model to push down a score it should be pushing up.

In [ ]:
results.to_csv(DATA / "comparison_results.csv")
print("saved -> data/comparison_results.csv (Notebook 04 appends Two-tower to this)")

---
# Section 4 — Retrieval: Two-tower
*(cells from `notebooks/04_retrieval_twotower.ipynb`)*

# 04 — Retrieval: Two-Tower

**Owner:** Person C · **Course session:** S3 (Retrieval at Scale) · **25% of grade with Notebook 03 · heaviest component**

Trains a two-tower retrieval model from scratch (`src/models/two_tower.py`) with in-batch
negatives + softmax cross-entropy, tunes on **val only**, evaluates against Popularity and
MF-BPR, and saves checkpoints for Notebook 05 (LightGBM) and the webapp to reload.

> **Colab setup:** Runtime → Change runtime type → **GPU (T4)**. This is the heaviest required
> component (10-15h across the team) — start it early. Mount Drive and save
> `data/two_tower_checkpoint.pt` there right after training so a disconnect doesn't cost a
> re-run.

**Before the defense, every team member must be able to explain (1) why in-batch negatives and
(2) why softmax cross-entropy over the batch** — both called out explicitly in
`src/models/two_tower.py`'s docstring as defense questions.

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS: torch+faiss both bundle OpenMP and abort on double-init otherwise
import sys, time, json
from pathlib import Path
sys.path.insert(0, ".." if Path("../src").exists() else ".")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config as C
from src.models.two_tower import TwoTower, make_history_batches, export_item_embeddings, user_vector
from src.retrieval.faiss_index import EmbeddingIndex
from src.eval.harness import evaluate_index, two_tower_user_vecs
from src.data.split import ground_truth_from

plt.rcParams["figure.dpi"] = 110
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

DATA = Path("../data") if Path("../data").exists() else Path("data")
DATA.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DATA = Path('/content/drive/MyDrive/amazon-recsys-data/data')
    if (DRIVE_DATA / "train.parquet").exists():
        import shutil
        shutil.copytree(DRIVE_DATA, DATA, dirs_exist_ok=True)
        print(f"restored data/ from Google Drive ({DRIVE_DATA}) -- this takes priority over "
              f"whatever a fresh git clone brought in, since Drive holds the last real run's "
              f"actual output, not just whatever was last committed to GitHub")
except ImportError:
    pass   # not running in Colab (e.g. local smoke-test)

train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
test = pd.read_parquet(DATA / "test.parquet")
ids = json.load(open(DATA / "id_encoders.json"))
user_ids, item_ids = ids["user_ids"], ids["item_ids"]
uid2u = {u: i for i, u in enumerate(user_ids)}
n_users, n_items = len(user_ids), len(item_ids)
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}  n_users={n_users:,}  n_items={n_items:,}")

comparison_results = pd.read_csv(DATA / "comparison_results.csv", index_col=0)
comparison_results

## Id spaces (same convention as Notebook 03)

Encoded ints (`u`/`i`) only to index into the model's embedding tables / build a history
sequence; everything eval-related (`ground_truth_from`, `seen_items`, `evaluate`, FAISS
labels) stays in raw string id space. `n_items + 1` reserves the trailing `<UNK>` row from
`encode_ids`, though in practice this eval never needs it: cold (train-unseen) test users are
simply excluded — the whole point of the cold-start analysis in Notebook 05 is to quantify how
much that costs, not to paper over it here.

## Training loop (with loss history) + a batched helper

`make_history_batches` sorts each user's train interactions by `timestamp` and, for every
`(user, target)` pair, uses only *earlier* items as history — the model never sees the target
or anything after it. This duplicates `src.models.two_tower.train_two_tower`'s loop with two
additions: a `temperature` argument (needed for the val-only tuning grid below — the shipped
`train_two_tower` hardcodes the model default) and loss history for plotting.

In [ ]:
def train_two_tower_with_history(train_df, n_items_total, dim, temperature, lr, epochs,
                                 batch_size, max_hist, device):
    model = TwoTower(n_items_total, dim, temperature=temperature).to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = []
    for ep in range(epochs):
        model.train(); running = n = 0
        for hist, mask, pos in make_history_batches(train_df, max_hist, batch_size, seed=ep):
            hist = torch.as_tensor(hist, device=device)
            mask = torch.as_tensor(mask, device=device)
            pos = torch.as_tensor(pos, device=device)
            opt.zero_grad()
            loss = model.in_batch_loss(hist, mask, pos)
            loss.backward(); opt.step()
            running += loss.item(); n += 1
        history.append(running / max(n, 1))
        print(f"epoch {ep+1}/{epochs}  loss={history[-1]:.4f}")
    return model, history


def two_tower_retrieval_metrics(model, eval_df, train_df, catalog_size, k_values, max_hist):
    """Builds each evaluated user's vector from their TRAIN-period history (not
    test-period) so this is a fair, leakage-free train->predict-future-test comparison
    against MF-BPR/Popularity. (At serving time the API does the opposite — runs the user
    tower over TEST-period history, i.e. the user's most recent real activity, to predict
    what they'll want *next*. Both are correct; they answer different questions.)"""
    gt = ground_truth_from(eval_df, positive_only=True)
    users = [uid for uid in gt if uid in uid2u]                     # drop cold (train-unseen) users
    train_hist = train_df.sort_values("timestamp").groupby("u")["i"].agg(list).to_dict()
    histories = [train_hist.get(uid2u[uid], []) for uid in users]
    uvecs = two_tower_user_vecs(model, histories, max_hist=max_hist, device=device_of(model))
    item_emb = export_item_embeddings(model, n_items + 1, device=device_of(model))[:n_items]
    index = EmbeddingIndex(item_emb, item_ids)
    return evaluate_index(index, uvecs, users, gt, train_df, catalog_size, k_values)


def device_of(model):
    return next(model.parameters()).device

## Hyperparameter search (val only)

Grid over `temperature` / `dim` / `max_hist`, each run a single short epoch — enough to rank
configs directionally, not to converge. **Never touches `test` here.**

In [ ]:
grid = [
    dict(temperature=0.05, dim=64, max_hist=20),
    dict(temperature=0.10, dim=64, max_hist=20),
    dict(temperature=0.05, dim=64, max_hist=10),
    dict(temperature=0.05, dim=32, max_hist=20),
]
search_rows = []
for cfg in grid:
    m, _ = train_two_tower_with_history(train, n_items + 1, cfg["dim"], cfg["temperature"],
                                        lr=1e-3, epochs=1, batch_size=1024,
                                        max_hist=cfg["max_hist"], device=DEVICE)
    _, m20 = two_tower_retrieval_metrics(m, val, train, catalog_size=n_items,
                                         k_values=(20,), max_hist=cfg["max_hist"])
    search_rows.append({**cfg, "val_Recall@20": m20["Recall@20"]})

search_df = pd.DataFrame(search_rows).sort_values("val_Recall@20", ascending=False)
search_df

**Pick the best row above** and train the final model longer (`EPOCHS=10`, matching
`src/models/two_tower.py`'s own default — the search only had to rank configs).

In [ ]:
BEST = search_df.iloc[0][["temperature", "dim", "max_hist"]].to_dict()
BEST["dim"] = int(BEST["dim"]); BEST["max_hist"] = int(BEST["max_hist"])
EPOCHS = 10
print("training with:", BEST)

t0 = time.time()
model, loss_history = train_two_tower_with_history(train, n_items + 1, BEST["dim"],
                                                    BEST["temperature"], lr=1e-3, epochs=EPOCHS,
                                                    batch_size=1024, max_hist=BEST["max_hist"],
                                                    device=DEVICE)
print(f"trained in {time.time()-t0:.0f}s")

fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(range(1, len(loss_history) + 1), loss_history, color="#a05d3b", marker="o")
ax.set_xlabel("epoch"); ax.set_ylabel("mean in-batch softmax CE loss"); ax.set_title("Two-tower training loss")
plt.tight_layout(); plt.show()

## Retrieval evaluation (test) — full comparison table

Popularity vs MF-BPR vs Two-tower, side by side — required in both `README.md` and
`ANALYSIS.md`.

In [ ]:
recs, tt_metrics = two_tower_retrieval_metrics(model, test, train, catalog_size=n_items,
                                               k_values=C.K_VALUES, max_hist=BEST["max_hist"])
tt_metrics

In [ ]:
comparison_results.loc["Two-tower"] = {k: v for k, v in tt_metrics.items() if k != "n_users_scored"}
comparison_results.round(4)

## Interpretation

Two-tower should edge out MF-BPR on Recall/NDCG — it conditions on a user's actual recent
history (not just a fixed learned embedding), and in-batch negatives give a denser training
signal per step than BPR's one-negative-at-a-time sampling. If it doesn't win outright here,
note that honestly — the dataset is small and sparse at this sample size (see Notebook 01),
and "best" is whatever the numbers above actually say, not a foregone conclusion. Either way,
Two-tower is what gets deployed (next section) regardless of which model wins this table,
because of how it's built, not because it won.

## Why Two-tower serves in production regardless of which model wins above

`api/recommender.py`'s `_load_tower` loads a **TorchScripted user tower** and forward-passes
a user's live TEST-period history through it at request time — that's how the API handles a
user whose latest activity the model never trained on without needing to retrain. MF-BPR has
no equivalent: its "user vector" is one fixed row in a lookup table, learned only for the
exact users seen during training, with no way to incorporate a user's later activity short of
retraining from scratch. So MF-BPR (Notebook 03) is a required, independently defensible
implementation and comparison-table entry, but the two-tower model here is what
`scripts/export_artifacts.py` scripts and what the API actually serves — this is a real
architectural constraint, not a preference.

## Precompute similar items (for "Similar items" and "Because you liked X")

Static — every item's nearest neighbours in embedding space, computed once here and served
straight from JSON at request time (`api/recommender.py`'s `.similar()` never touches FAISS
live).

In [ ]:
item_emb_full = export_item_embeddings(model, n_items + 1, device=DEVICE)
item_emb_real = item_emb_full[:n_items]                       # drop the <UNK> row
index = EmbeddingIndex(item_emb_real, item_ids)

t0 = time.time()
similar_map = {iid: index.similar_items(iid, n=10) for iid in item_ids}
print(f"precomputed similar_items for {len(similar_map):,} items in {time.time()-t0:.0f}s")
list(similar_map.items())[:2]

## Save checkpoint + embeddings for Notebook 05

Each notebook here is its own Colab session, so we checkpoint everything Notebook 05 needs to
reload rather than assume shared memory — same reasoning as freezing the split in Notebook 01.
Save these to Drive in the real run so a disconnect doesn't cost the 10-15h of work.

In [ ]:
torch.save({
    "state_dict": model.state_dict(),
    "n_items_total": n_items + 1,
    "dim": BEST["dim"],
    "temperature": BEST["temperature"],
    "max_hist": BEST["max_hist"],
}, DATA / "two_tower_checkpoint.pt")

np.save(DATA / "item_emb.npy", item_emb_real)
json.dump(similar_map, open(DATA / "similar_items.json", "w"))
comparison_results.to_csv(DATA / "comparison_results.csv")
print("saved -> data/{two_tower_checkpoint.pt, item_emb.npy, similar_items.json, comparison_results.csv}")

---
# Section 5 — Ranking: LightGBM + artifact export
*(cells from `notebooks/05_ranking_lightgbm.ipynb`)*

# 05 — Ranking: LightGBM (LambdaRank)

**Owner:** Person A + everyone on features · **Course session:** S4 (Ranking Models) · **15% of grade**

Builds a re-ranker on top of the two-tower retriever (the deployed retriever — see Notebook
04's "why two-tower serves in production" section), trains LightGBM with the LambdaRank
objective on 12 features, reports the retrieval-only vs +ranking ablation, plots feature
importance, runs the cold-start analysis for `ANALYSIS.md`, and finally exports everything the
API needs via `scripts/export_artifacts.py`.

> **Colab setup:** CPU runtime is fine — neither FAISS search nor LightGBM training need a
> GPU (save GPU quota; only Notebooks 03/04 need it). Load Notebook 04's checkpoint from Drive.

**Leakage-safe design (own it at the defense):** candidates are retrieved for **train** users
using their **train**-period history, computed features come from **train** only, but a
candidate is labeled positive if it appears in that user's **validation**-period ground truth
— never train. This teaches the ranker to predict *future* engagement from *past* signal,
matching deployment, instead of memorizing train positives (`src/ranking/pipeline.py`).

In [ ]:
import os
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # macOS: torch+faiss both bundle OpenMP and abort on double-init otherwise
import sys, time, json, pickle
from pathlib import Path
sys.path.insert(0, ".." if Path("../src").exists() else ".")

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

from src import config as C
from src.models.two_tower import TwoTower, export_item_embeddings
from src.retrieval.faiss_index import EmbeddingIndex
from src.eval.harness import seen_items, two_tower_user_vecs
from src.eval.metrics import evaluate
from src.eval.cold_start import evaluate_by_bucket
from src.data.split import ground_truth_from
from src.baselines.popularity import PopularityRecommender
from src.ranking.features import build_feature_store, featurize, FEATURE_NAMES
from src.ranking.pipeline import build_ranker_data
from src.ranking.ranker import train_ranker, rerank, feature_importance
import scripts.export_artifacts as export_artifacts

plt.rcParams["figure.dpi"] = 110
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

DATA = Path("../data") if Path("../data").exists() else Path("data")
DATA.mkdir(parents=True, exist_ok=True)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    DRIVE_DATA = Path('/content/drive/MyDrive/amazon-recsys-data/data')
    if (DRIVE_DATA / "train.parquet").exists():
        import shutil
        shutil.copytree(DRIVE_DATA, DATA, dirs_exist_ok=True)
        print(f"restored data/ from Google Drive ({DRIVE_DATA}) -- this takes priority over "
              f"whatever a fresh git clone brought in, since Drive holds the last real run's "
              f"actual output, not just whatever was last committed to GitHub. This must be "
              f"the SAME data/two_tower_checkpoint.pt Notebook 04 trained, or everything below "
              f"is internally inconsistent.")
except ImportError:
    pass   # not running in Colab (e.g. local smoke-test)

train = pd.read_parquet(DATA / "train.parquet")
val = pd.read_parquet(DATA / "val.parquet")
test = pd.read_parquet(DATA / "test.parquet")
items = pd.read_parquet(DATA / "items.parquet")
ids = json.load(open(DATA / "id_encoders.json"))
user_ids, item_ids = ids["user_ids"], ids["item_ids"]
uid2u = {u: i for i, u in enumerate(user_ids)}
n_users, n_items = len(user_ids), len(item_ids)
comparison_results = pd.read_csv(DATA / "comparison_results.csv", index_col=0)
print(f"train={len(train):,}  val={len(val):,}  test={len(test):,}")
comparison_results.round(4)

## Reload the two-tower retriever (Notebook 04's checkpoint)

Every notebook here is its own Colab session — reload rather than assume shared memory.

In [ ]:
ckpt = torch.load(DATA / "two_tower_checkpoint.pt", map_location=DEVICE)
tt_model = TwoTower(ckpt["n_items_total"], ckpt["dim"], temperature=ckpt["temperature"]).to(DEVICE)
tt_model.load_state_dict(ckpt["state_dict"])
tt_model.eval()
MAX_HIST = ckpt["max_hist"]

item_emb = np.load(DATA / "item_emb.npy")                 # already excludes the <UNK> row
retrieval_index = EmbeddingIndex(item_emb, item_ids)
similar_map = json.load(open(DATA / "similar_items.json"))
print(f"reloaded two-tower (dim={ckpt['dim']}, temperature={ckpt['temperature']}, max_hist={MAX_HIST})")

## Retrieve candidates for train users

Top-100 per train user (train-seen items excluded), built from the user's **train**-period
history — the same protocol every retrieval eval in this project uses, so candidate quality
here is directly comparable to Notebook 04's reported test metrics.

In [ ]:
def batch_retrieve(users_raw, history_source_df, index, n=100, max_hist=MAX_HIST):
    """Returns (candidates, scores): {raw_user_id -> [item_id,...]} train-seen excluded,
    and {(raw_user_id, item_id) -> similarity} for the featurizer's retriever_score."""
    hist_by_u = history_source_df.sort_values("timestamp").groupby("u")["i"].agg(list).to_dict()
    known = [u for u in users_raw if u in uid2u]
    histories = [hist_by_u.get(uid2u[u], []) for u in known]
    uvecs = two_tower_user_vecs(tt_model, histories, max_hist=max_hist, device=DEVICE)
    seen = seen_items(train)
    q = uvecs.astype("float32")
    import faiss; faiss.normalize_L2(q)
    D, I = retrieval_index.index.search(q, n * 2)          # over-retrieve, then drop train-seen
    cands, scores = {}, {}
    for row, u in enumerate(known):
        s = seen.get(u, set())
        kept = 0
        for col in range(I.shape[1]):
            it = str(retrieval_index.item_ids[I[row, col]])
            if it in s:
                continue
            cands.setdefault(u, []).append(it)
            scores[(u, it)] = float(D[row, col])
            kept += 1
            if kept >= n:
                break
    return cands, scores

train_users_raw = train["user_id"].unique().tolist()
t0 = time.time()
candidates, retriever_scores = batch_retrieve(train_users_raw, train, retrieval_index, n=100)
print(f"retrieved candidates for {len(candidates):,} / {len(train_users_raw):,} train users "
      f"in {time.time()-t0:.0f}s")

## Feature store + leakage-safe labels

`build_feature_store` computes every aggregate from **train only**. Labels come from the
**validation**-period ground truth — a candidate is positive if the user actually interacted
with it there, negative otherwise. `build_ranker_data` drops any user whose top-100 candidates
contain zero validation positives (LambdaRank needs at least one positive per query group).

In [ ]:
store = build_feature_store(train, items)
now_ts = int(train["timestamp"].max())
val_gt = ground_truth_from(val, positive_only=True)

rng = np.random.default_rng(C.SEED)
cand_users = list(candidates.keys())
rng.shuffle(cand_users)
split_at = int(0.85 * len(cand_users))
ranker_train_users, ranker_val_users = cand_users[:split_at], cand_users[split_at:]
print(f"ranker-internal split (for LightGBM early stopping only, distinct from the "
      f"project train/val/test split above): {len(ranker_train_users):,} / {len(ranker_val_users):,} users")

def subset(users):
    return {u: candidates[u] for u in users}

X_train, y_train, group_train, _ = build_ranker_data(subset(ranker_train_users), val_gt, store, now_ts, retriever_scores)
X_val, y_val, group_val, _ = build_ranker_data(subset(ranker_val_users), val_gt, store, now_ts, retriever_scores)
print(f"ranker rows: train={len(X_train):,} (groups={len(group_train)})  val={len(X_val):,} (groups={len(group_val)})")
val_pos_rate = y_val.mean() if len(y_val) else float("nan")
print(f"positive rate: train={y_train.mean():.3f}  val={val_pos_rate:.3f}")

## Train LightGBM (LambdaRank)

In [ ]:
if len(X_val) == 0:
    print("WARNING: ranker-val candidate set has zero rows with a validation-period positive "
          "overlap (small-sample artifact) -- training without a held-out early-stopping set.")
    lgbm_model = train_ranker(X_train, y_train, group_train)
else:
    lgbm_model = train_ranker(X_train, y_train, group_train, X_val, y_val, group_val)

## Ablation: retrieval-only vs +ranking (test)

Retrieval-only = Notebook 04's Two-tower test row (already in `comparison_results`).
+Ranking = retrieve top-100 for **test** users the same way, rerank with LightGBM, keep top-10,
evaluate against test-period ground truth.

In [ ]:
test_users_raw = [u for u in test["user_id"].unique() if u in uid2u]
test_cands, test_scores = batch_retrieve(test_users_raw, train, retrieval_index, n=100)

def rerank_user(u, cand_items):
    df = pd.DataFrame({"user_id": [u] * len(cand_items), "item_id": cand_items,
                       "retriever_score": [test_scores.get((u, it), 0.0) for it in cand_items]})
    feats = featurize(df, store, now_ts)
    top_items, _ = rerank(lgbm_model, feats, cand_items, top_k=max(C.K_VALUES))
    return top_items

t0 = time.time()
lgbm_recs = {u: rerank_user(u, items_) for u, items_ in test_cands.items()}
print(f"reranked {len(lgbm_recs):,} test users in {time.time()-t0:.0f}s")

test_gt = ground_truth_from(test, positive_only=True)
lgbm_metrics = evaluate(lgbm_recs, test_gt, catalog_size=n_items, k_values=C.K_VALUES)
lgbm_metrics

In [ ]:
tt_metrics = evaluate(test_cands, test_gt, catalog_size=n_items, k_values=C.K_VALUES)
comparison_results.loc["Two-tower"] = {
    "Recall@10": tt_metrics["Recall@10"],
    "NDCG@10": tt_metrics["NDCG@10"],
    "Recall@20": tt_metrics["Recall@20"],
    "NDCG@20": tt_metrics["NDCG@20"],
    "Recall@50": tt_metrics["Recall@50"],
    "NDCG@50": tt_metrics["NDCG@50"],
    "Coverage@50": tt_metrics.get("Coverage@50", np.nan),
}
display(comparison_results.round(4))

In [ ]:
ablation = pd.DataFrame([
    {"setup": "Two-tower only (retrieval)",
     "NDCG@10": comparison_results.loc["Two-tower", "NDCG@10"],
     "Coverage@50": comparison_results.loc["Two-tower", "Coverage@50"]},
    {"setup": "Two-tower + LightGBM",
     "NDCG@10": lgbm_metrics["NDCG@10"], "Coverage@50": lgbm_metrics.get("Coverage@50", np.nan)},
]).set_index("setup")
ablation.round(4)

**Interpretation.** Re-ranking should raise NDCG@10 — LightGBM sees per-candidate
features (popularity, price, category/brand match, recency) that pure embedding similarity
can't express directly — typically at some cost to Coverage, since the ranker learns to
prefer generally-engaging candidates (higher `item_pop`, matching categories) over purely
novel ones. State the actual trade-off observed here in `ANALYSIS.md` §2, not the expected
direction — if Coverage barely moves, that's worth noting too (means the 12 features aren't
just reinventing popularity).

In [ ]:
comparison_results.loc["Two-tower + LightGBM"] = {
    k: v for k, v in lgbm_metrics.items() if k != "n_users_scored"
}
comparison_results.round(4)

## Feature importance

In [ ]:
imp = feature_importance(lgbm_model)
imp_df = pd.DataFrame(imp, columns=["feature", "gain"])
fig, ax = plt.subplots(figsize=(6, 4))
ax.barh(imp_df["feature"][::-1], imp_df["gain"][::-1], color="#3b6fa0")
ax.set_xlabel("gain"); ax.set_title("LightGBM feature importance")
plt.tight_layout(); plt.show()
imp_df

## Cold-start analysis (`ANALYSIS.md` §3)

Bucket test users by their **train**-period activity, compare Popularity vs Two-tower per
bucket. Expect Popularity to dominate (or tie) in the coldest bucket — a near-empty history
gives the two-tower user tower almost nothing to pool — and Two-tower to pull ahead as users
get more active train history to build a real vector from.

In [ ]:
pop_model = PopularityRecommender().fit(train)
pop_recs = pop_model.recommend(list(test_gt.keys()), n=max(C.K_VALUES), exclude=seen_items(train))

tt_train_hist = train.sort_values("timestamp").groupby("u")["i"].agg(list).to_dict()
tt_known_users = [u for u in test_gt if u in uid2u]
tt_histories = [tt_train_hist.get(uid2u[u], []) for u in tt_known_users]
tt_uvecs = two_tower_user_vecs(tt_model, tt_histories, max_hist=MAX_HIST, device=DEVICE)
from src.eval.harness import recommend_from_index
tt_recs = recommend_from_index(retrieval_index, tt_uvecs, tt_known_users, seen_items(train), n=max(C.K_VALUES))

pop_buckets = evaluate_by_bucket(pop_recs, test_gt, train, k=20)
tt_buckets = evaluate_by_bucket(tt_recs, test_gt, train, k=20)
cold_start = pop_buckets.merge(tt_buckets, on=["bucket", "users"], suffixes=(" (pop)", " (two-tower)"))
cold_start

## Export everything the API needs

`scripts/export_artifacts.py` writes `artifacts/` (mounted read-only into the API container by
`docker-compose.yml`). Popularity list is refit here (deterministic, cheap) rather than
reloaded, since Notebook 02 didn't persist the fitted model object, only its metrics.

In [ ]:
ARTIFACTS = Path("../artifacts") if Path("../artifacts").exists() else Path("artifacts")
popularity_ids = pop_model.ranked_items

export_artifacts.export(
    popularity_ids=popularity_ids,
    item_encoder=item_ids,
    item_emb=item_emb,
    two_tower_model=tt_model,
    lgbm_model=lgbm_model,
    similar_map=similar_map,
    feature_store=store,
    now_ts=now_ts,
)
pickle.dump(lgbm_model, open(DATA / "lgbm.pkl", "wb"))
comparison_results.to_csv(DATA / "comparison_results.csv")
ablation.to_csv(DATA / "ablation_results.csv")
cold_start.to_csv(DATA / "cold_start_results.csv", index=False)
imp_df.to_csv(DATA / "feature_importance.csv", index=False)
print("exported -> ../artifacts/  (copy this whole folder next to docker-compose.yml to serve it)")
print("saved -> data/{comparison_results,ablation_results,cold_start_results,feature_importance}.csv")

## Summary

Final comparison table (`comparison_results`), the retrieval-vs-+ranking ablation
(`ablation`), the cold-start breakdown (`cold_start`), and feature importance (`imp_df`) are
all saved to `data/*.csv` — Section 8 (README/ANALYSIS.md) reads these directly rather than
re-running any notebook. Next: Step 7 in `PROJECT_GUIDE.md` — bring up the webapp against
`../artifacts/` and run `scripts/benchmark_latency.py` for the latency table.

---
# Sync results back to Drive

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    import os, shutil
    DRIVE_DIR = '/content/drive/MyDrive/amazon-recsys-data'
    for d in ["data", "artifacts"]:
        if os.path.isdir(d):
            shutil.copytree(d, os.path.join(DRIVE_DIR, d if d != "data" else "data"), dirs_exist_ok=True)
            print(f"synced {d}/ -> {DRIVE_DIR}/{d}")
except ImportError:
    print("not in Colab — skipping Drive sync")